# Notebook 04 — Full Production Simulation & Analysis Pipeline

This notebook is a **1-to-1 consolidated mirror** of the production shell script [`scripts/run_full_production_simulation.sh`](../scripts/run_full_production_simulation.sh).

## Objectives & Features
1. **Input Hash Verification**: Cryptographic validation of `By.txt`, `kickmap_file.txt`, `K4GSR_HBIv4-1.mat`.
2. **NKM Field Validation**: 5-way cross-validation of 1D longitudinal profiles and 2D horizontal kickmaps.
3. **Symplectic Slicing Convergence**: Scans $N_{\text{slices}} \in \{5, 10, 20, 50, 100\}$.
4. **Multi-Turn Storage Ring Injection**: Tracks 1000 particles over 1000 turns across 4 kicker models with physical apertures.
5. **Deterministic BTS Quad Matching**: 2-stage SLSQP optics optimization & SVD Jacobian analysis.
6. **Monte Carlo Tolerance Budget**: 500-seed sensitivity study executed across ~90% CPU cores.
7. **MOGA Pareto Study**: Multi-seed NSGA-II Pareto trade-off optimization.
8. **Data-Driven Publication Summary**: Compiles summary figures, LaTeX tables, and provenance logs.

All outputs are saved under a structured result directory: `results/production_run_<timestamp>/`.

In [ ]:
import sys
import os
import datetime
from pathlib import Path

# Set repository root
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Determine parallel worker cores (90% of available CPU cores)
total_cores = os.cpu_count() or 4
n_workers = max(1, int(total_cores * 0.9))

# Create structured output directories
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = repo_root / "results" / f"production_run_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"=== NKM Full Production Simulation Pipeline ===")
print(f"Timestamp       : {timestamp}")
print(f"Total CPU Cores : {total_cores}")
print(f"Allocated Cores : {n_workers} (~90% CPU capacity)")
print(f"Output Directory: {output_dir}")

## Step 1: Input Hash Cataloging & Baseline Metrics Verification
Verifies that all scientific source files remain untampered and records baseline metrics.

In [ ]:
from scripts.inventory_protected_hashes import create_hash_manifest, verify_hash_manifest, OUTPUT_MANIFEST
from scripts.record_baseline_metrics import record_baseline_metrics

manifest = create_hash_manifest()
print(f"[Step 1] Cataloged {len(manifest)} protected scientific source files.")
for path, h in manifest.items():
    print(f"  {path:30s} -> {h[:16]}...")

assert verify_hash_manifest(OUTPUT_MANIFEST), "Protected file verification failed!"
record_baseline_metrics()
print("[Step 1] Baseline metrics recorded successfully.")

## Step 2: NKM Field & Kick Map Cross-Validation
Parses 1D longitudinal profiles and 2D horizontal kickmaps, fits 5th-order field polynomials, and checks symmetry residuals.

In [ ]:
from scripts.validate_nkm_fieldmap import run_fieldmap_validation
from scripts.validate_nkm_kick import run_nkm_kick_validation

print("[Step 2] Executing fieldmap validation...")
run_fieldmap_validation()
run_nkm_kick_validation()
print("[Step 2] Field and Kick Map Cross-Validation Complete.")

## Step 3: Symplectic Slicing Convergence Scan
Evaluates thick particle tracking integration convergence across $N_{\text{slices}} \in \{5, 10, 20, 50, 100\}$.

In [ ]:
from scripts.run_tracking_convergence import run_slicing_convergence_study

print("[Step 3] Executing slicing convergence scan...")
run_slicing_convergence_study()
print("[Step 3] Slicing Convergence Scan Complete.")

## Step 4: Multi-Turn Storage Ring Injection Tracking
Simulates 1000 particles over 1000 turns across 4 kicker models with physical apertures.

In [ ]:
from scripts.run_multiturn_injection import main as run_multiturn

print("[Step 4] Simulating multi-turn storage ring injection dynamics...")
run_multiturn()
print("[Step 4] Multi-Turn Injection Simulation Complete.")

## Step 5: Deterministic BTS Quadrupole Matching
Executes 2-stage SLSQP quadrupole optimization for target Twiss matching ($\beta_x=7.56\text{ m}, \beta_y=12.27\text{ m}$) and SVD Jacobian analysis.

In [ ]:
from scripts.optimize_bts_publication import run_bts_publication_optimization

print("[Step 5] Executing BTS Quadrupole Matching...")
run_bts_publication_optimization()
print("[Step 5] BTS Quadrupole Optics Matching Complete.")

## Step 6: Monte Carlo Tolerance Budget & Sensitivity Analysis
Evaluates 500 Monte Carlo seeds with quad errors using ~90% CPU cores.

In [ ]:
from scripts.run_publication_tolerances import run_publication_tolerance_study

print(f"[Step 6] Running Monte Carlo tolerance budget using {n_workers} CPU cores...")
run_publication_tolerance_study()
print("[Step 6] Monte Carlo Tolerance Budget Complete.")

## Step 7: Multi-Objective Pareto Study (MOGA)
NSGA-II multi-objective optimization balancing mismatch, beta peak, and stay-clears.

In [ ]:
from scripts.run_publication_moga import main as run_moga

print("[Step 7] Running MOGA Pareto Optimization Study...")
run_moga()
print("[Step 7] MOGA Pareto Study Complete.")

## Step 8: Publication Data Consolidation & Figure Generation
Compiles final publication figures, tables, and metric summaries.

In [ ]:
from scripts.reproduce_paper import main as reproduce_paper

print("[Step 8] Compiling publication figures and metric summary...")
reproduce_paper()
print("=== Full Production Simulation & Analysis Pipeline Complete! ===")